# XTTS v2 Korean Fine-Tuning · Colab Free 최적화

**환경**: T4 (15GB VRAM) · 12시간 세션 한계 · idle 90분 disconnect

**전략**:
- 데이터 5,000건만 사용 (전체 12K 중)
- batch_size 2 + grad_accum 4 = effective batch 8
- 3 epoch (5 → 3)
- 500 step마다 체크포인트 저장 → Drive
- 끊겨도 재시작 셀로 복귀 가능
- Anti-idle JS 트릭 적용

**필요**:
- Google Drive에 `xtts_korean/{kss, manifest}` 업로드
- KSS는 ~3GB, manifest는 수 MB

In [ ]:
# 0. GPU 확인 — T4가 잡혔는지
!nvidia-smi | head -20

In [ ]:
# 1. Anti-idle (90분 끊김 방지)
# 브라우저 콘솔에 자동 클릭 JS 띄움
from IPython.display import display, Javascript
display(Javascript('''
  function clickConnect(){
    const btn = document.querySelector('colab-toolbar-button#connect') || document.querySelector('paper-icon-button');
    if (btn) btn.click();
  }
  setInterval(clickConnect, 60000);
'''))
print('Anti-idle JS installed.')

In [ ]:
# 2. 의존성 설치 (Coqui fork)
!pip install -q coqui-tts==0.24.3 trainer pandas soundfile librosa
import TTS, torch
print('TTS:', TTS.__version__, '· CUDA:', torch.cuda.is_available())

In [ ]:
# 3. Drive 마운트 + 경로 설정
from google.colab import drive
drive.mount('/content/drive')

import os, glob
DRIVE_ROOT = '/content/drive/MyDrive/xtts_korean'
DATASET    = f'{DRIVE_ROOT}/kss'
MANIFEST   = f'{DRIVE_ROOT}/manifest'
OUTPUT     = f'{DRIVE_ROOT}/runs'
os.makedirs(OUTPUT, exist_ok=True)

assert os.path.exists(f'{MANIFEST}/metadata_train.csv'), 'manifest 없음'
assert glob.glob(f'{DATASET}/**/*.wav', recursive=True)[:1], 'KSS WAV 없음'
print('train rows :', sum(1 for _ in open(f'{MANIFEST}/metadata_train.csv', encoding='utf-8')))
print('val rows   :', sum(1 for _ in open(f'{MANIFEST}/metadata_val.csv', encoding='utf-8')))

In [ ]:
# 4. Pretrained XTTS v2 다운로드 (한 번만 — 캐시됨)
from TTS.utils.manage import ModelManager
mm = ModelManager()
model_path, config_path, _ = mm.download_model('tts_models/multilingual/multi-dataset/xtts_v2')
PRETRAINED_DIR = os.path.dirname(model_path)
print('Pretrained:', PRETRAINED_DIR)

In [ ]:
# 5. Fine-tune Config (Free 환경 최적화)
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.config.shared_configs import BaseDatasetConfig

config = XttsConfig()
config.load_json(config_path)

# ── Free 최적화 ──
config.epochs           = 3
config.batch_size       = 2
config.eval_batch_size  = 1
config.grad_accum_steps = 4   # effective batch = 8
config.mixed_precision  = True  # T4 VRAM 절약
config.lr               = 5e-6
config.optimizer        = 'AdamW'
config.optimizer_params = {'betas': [0.9, 0.96], 'eps': 1e-8, 'weight_decay': 1e-2}
config.save_step        = 500
config.save_n_checkpoints = 2    # Drive 용량 절약 (체크포인트 7GB×2)
config.print_step       = 50
config.run_name         = 'xtts_korean_kss_free'
config.output_path      = OUTPUT

dataset = BaseDatasetConfig(
    formatter='ljspeech',
    meta_file_train=f'{MANIFEST}/metadata_train.csv',
    meta_file_val=f'{MANIFEST}/metadata_val.csv',
    path=DATASET, language='ko',
)
config.datasets = [dataset]
print('Config ready · batch={} · effective={} · epochs={}'.format(
    config.batch_size, config.batch_size * config.grad_accum_steps, config.epochs))

In [ ]:
# 6. 모델 초기화 + 체크포인트 resume 자동 감지
from trainer import Trainer, TrainerArgs
from TTS.tts.models.xtts import Xtts

model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_dir=PRETRAINED_DIR)

# 끊겼다 재시작하는 경우 — Drive에서 가장 최근 체크포인트 찾기
restore = None
ckpts = sorted(glob.glob(f'{OUTPUT}/*/checkpoint_*.pth'))
if ckpts:
    restore = ckpts[-1]
    print(f'[RESUME] {restore}')
else:
    print('[FRESH] no checkpoint found · 처음부터 학습')

trainer = Trainer(
    TrainerArgs(restore_path=restore),
    config, output_path=OUTPUT, model=model,
)

In [ ]:
# 7. 학습 실행
# Free T4 기준 5,000 샘플 × 3 epoch ≈ 6~9 시간
# 끊기면 셀 6번부터 재실행 → 자동 resume
trainer.fit()

In [ ]:
# 8. Best checkpoint 확인
import glob
best = sorted(glob.glob(f'{OUTPUT}/*/best_model.pth'))
print('Best model :', best[-1] if best else 'not found')
print('All ckpts  :', sorted(glob.glob(f'{OUTPUT}/*/*.pth'))[-3:])

## 학습 후 평가
1. `best_model.pth`를 로컬로 다운로드 (또는 Drive 그대로 마운트)
2. `tuning/3_eval/run_eval.py --model finetuned --ckpt best_model.pth ...`
3. `compare.py`로 Base와 비교

## Tips
- 세션 끊기면 셀 6번부터 다시 실행 → 자동 resume
- Drive `runs/` 용량 모자라면 `save_n_checkpoints=1`로 줄이기
- T4 OOM 발생 시 `batch_size=1, grad_accum_steps=8`로 조정